# Download & Setup GTSRB Dataset\n\nDownloads the **German Traffic Sign Recognition Benchmark** (~50K images, 43 classes) and maps it to our 4 classes:\n- **crosswalk** (from: pedestrians, children crossing)\n- **speedlimit** (from: all speed limit signs)\n- **stop** (from: stop sign)\n- **trafficlight** (from: traffic signals)\n\nRun all cells, then use `train_models.ipynb` to train.

In [ ]:
import os, shutil
from pathlib import Path
from collections import Counter
from PIL import Image
from torchvision.datasets import GTSRB

# ── Config ──
OUTPUT_DIR = Path("data/GTSRB_mapped")
IMAGE_SIZE = 224
MAX_PER_CLASS = 2000  # cap to keep classes balanced

# ── GTSRB class ID → our class name ──
GTSRB_MAPPING = {
    0: "speedlimit",   # 20km/h
    1: "speedlimit",   # 30km/h
    2: "speedlimit",   # 50km/h
    3: "speedlimit",   # 60km/h
    4: "speedlimit",   # 70km/h
    5: "speedlimit",   # 80km/h
    7: "speedlimit",   # 100km/h
    8: "speedlimit",   # 120km/h
    14: "stop",
    26: "trafficlight",
    27: "crosswalk",    # Pedestrians
    28: "crosswalk",    # Children crossing
}

print(f"Output directory: {OUTPUT_DIR}")
print(f"Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Max per class: {MAX_PER_CLASS}")

## 1 — Download GTSRB (takes 1-2 min)

In [ ]:
print("Downloading GTSRB train split...")\ntrain_gtsrb = GTSRB(root="data/gtsrb_raw", split="train", download=True)\nprint(f"  Train: {len(train_gtsrb)} images")\n\nprint("Downloading GTSRB test split...")\ntest_gtsrb = GTSRB(root="data/gtsrb_raw", split="test", download=True)\nprint(f"  Test: {len(test_gtsrb)} images")\nprint("Download complete!")

## 2 — Map to 4 classes & save

In [ ]:
def process_split(dataset, split_name):
    """Map GTSRB classes to our 4 classes and save as 224x224 PNGs."""
    class_counts = Counter()
    saved = 0

    for idx in range(len(dataset)):
        img, gtsrb_label = dataset[idx]

        if gtsrb_label not in GTSRB_MAPPING:
            continue

        our_class = GTSRB_MAPPING[gtsrb_label]

        if class_counts[our_class] >= MAX_PER_CLASS:
            continue

        out_dir = OUTPUT_DIR / split_name / our_class
        out_dir.mkdir(parents=True, exist_ok=True)

        img = img.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        img.save(out_dir / f"{our_class}_{gtsrb_label}_{idx:06d}.png")

        class_counts[our_class] += 1
        saved += 1

        if saved % 1000 == 0:
            print(f"  [{split_name}] Saved {saved} images...")

    print(f"\n  {split_name} done: {saved} images total")
    for cls, cnt in sorted(class_counts.items()):
        print(f"    {cls}: {cnt}")
    return class_counts

print("Processing train split...")
train_counts = process_split(train_gtsrb, "train")

print("\nProcessing test split...")
test_counts = process_split(test_gtsrb, "test")

## 3 — Verify dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

classes = ["crosswalk", "speedlimit", "stop", "trafficlight"]

# Count images per class
print("=== Final Dataset Summary ===\n")
for split in ["train", "test"]:
    print(f"{split.upper()}:")
    total = 0
    for cls in classes:
        d = OUTPUT_DIR / split / cls
        n = len(list(d.glob("*.png"))) if d.exists() else 0
        total += n
        print(f"  {cls:15s}  {n:5d} images")
    print(f"  {'TOTAL':15s}  {total:5d} images\n")

# Show sample images
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, cls in enumerate(classes):
    cls_dir = OUTPUT_DIR / "train" / cls
    imgs = sorted(cls_dir.glob("*.png"))[:4]
    for j, img_path in enumerate(imgs):
        ax = axes[i // 2, (i % 2) * 4 + j]
        ax.imshow(Image.open(img_path))
        ax.set_title(cls, fontsize=8)
        ax.axis("off")
plt.suptitle("GTSRB Mapped Samples (224x224)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4 — How to use in training\n\nIn `train_models.ipynb` **cell 4**, change these 2 lines:\n\n```python\n# OLD (current dataset — 702 images)\nrecords = load_records("annotations", "images")\ntrain_records, val_records = stratified_split(records, val_ratio=0.2, seed=42)\n\n# NEW (GTSRB — ~8000 images)\nfrom road_sign_data import load_records_imagefolder\ntrain_records = load_records_imagefolder("data/GTSRB_mapped", split="train")\nval_records = load_records_imagefolder("data/GTSRB_mapped", split="test")\n```\n\nThen **Run All Cells** in `train_models.ipynb`. Everything else works as-is.